- fitz est le nom utilisé pour importer PyMuPDF.

In [62]:
import fitz

In [65]:
chemin_pdf = "../documents/caisse_des_depots.pdf"
document = fitz.open(chemin_pdf)

In [66]:
# nombre de page 
len(document)

92

In [67]:
# print(document[1].get_text())

In [68]:
liste_pages = [ ]

for page in document :

    texte = page.get_text()
    if texte.strip() != "":
        liste_pages.append(texte)

texte_complet = "\n".join(liste_pages)

In [69]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [70]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000 , 
    chunk_overlap = 100,
    separators=["\n\n" , "\n" , "." , "" , " "]
)


chunks = splitter.split_text(texte_complet)

len(chunks)

print(chunks[4])
print("\n")
print(chunks[5])

10
Les importants développements qu’elle a connu dans la
banque d’investissement la place  en concurrence avec des acteurs
privés de premier plan à l’échelon français et européen, ce qui ne
manque pas de susciter des interrogations croissantes.
Aujourd’hui, la Caisse des Dépôts est la tête d’un véritable
empire qui exerce à la fois des missions publiques et d’intérêt général
et des activités concurrentielles. Entre service public et activités
privées, la CDC est aujourd’hui à la croisée des chemins.
A l’heure du choix de nouveaux axes de développement pour
la CDC, dans un contexte de stabilité politique favorable aux
réformes, il nous est apparu nécessaire pour l’Etat d’engager une
réflexion pour faire le point sur les réorientations successives des
activités de la CDC.
Si les avis divergent sur le contenu d’une éventuelle réforme,
il existe en revanche un consensus général sur la nécessité de se
pencher sur l’avenir de la CDC. C’est pourquoi, nous tenions à


pencher sur l’avenir de l

In [71]:
len(chunks[1])

949

# emb

In [72]:
from dotenv import load_dotenv
import os 

load_dotenv()

Key = os.getenv("OPEN_AI_KEY")

from langchain_openai import OpenAIEmbeddings
emb =  OpenAIEmbeddings(api_key=Key , model="text-embedding-3-large")





In [73]:
phrases = [
    "Le RSA est une aide financière.",
    "Le revenu de solidarité active est une prestation sociale.",
    "J'aime manger une pizza." ,
]

In [74]:
vec1 = emb.embed_query(phrases[0])
vec2 = emb.embed_query(phrases[1])
vec3 = emb.embed_query(phrases[2])




In [75]:
vactor = emb.aembed_documents(phrases)

/var/folders/88/rj2_s1vs5dx5bqyvb1nrv3hw0000gn/T/ipykernel_45867/3338727581.py:1: RuntimeWarning: coroutine 'OpenAIEmbeddings.aembed_documents' was never awaited
  vactor = emb.aembed_documents(phrases)


In [78]:
from sklearn.metrics.pairwise import cosine_similarity 

In [79]:
cosine_similarity([vec1] , [vec2])

array([[0.65834524]])

In [80]:
cosine_similarity([vec1] , [vec3])

array([[0.18568147]])

#Faiss 

In [81]:
from langchain_community.vectorstores import FAISS

In [82]:
vectorstore = FAISS.from_texts(
    texts=chunks , 
    embedding=emb
)

In [83]:
vectorstore

In [105]:
question = " est ce que la caisse des depots recrute des alternant actuellement ?"

resultats = vectorstore.similarity_search(
    question,
    k=3
)

In [106]:
print(resultats[2].page_content)

75
économiques promulguée en mai 2001. Ces missions relèvent de
mandats qui lui sont confiés par l’Etat, les collectivités territoriales,
et plus récemment par l’Union européenne.
Ces missions s’organisent autour de trois métiers :
-  gestionnaire de fonds requérant une protection particulière :
la Caisse des dépôts gère des fonds d’épargne sur livrets exonérés
d’impôts ; des fonds déposés auprès de professions juridiques
(consignations, notaires, administrateurs et mandataires judiciaires)
et des caisses de retraites publiques ;
- prêteur public : elle finance par des prêts sur fonds
d’épargne des investissements d’intérêt général, principalement le
logement social locatif et le renouvellement urbain ;
- investisseur public : elle finance sur ses fonds propres des
investissements d’intérêt général en appui des politiques publiques
nationales et locales.
Récemment, les Pouvoirs publics ont étendu le financement
sur fonds d’épargne à de nouveaux secteurs d’intérêt général, et ont


In [107]:
contexte = "\n\n".join(

    doc.page_content for doc in resultats

)

In [108]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=Key , 
    model="gpt-4.1-mini", 
    temperature=0 
)

In [109]:
prompt = f"""
Tu es un assistant d'analyse documentaire.

Réponds à la question uniquement à partir du contexte fourni.
Si la réponse n'est pas présente dans le contexte, indique clairement
que l'information n'a pas été trouvée dans les documents.

Contexte :
{contexte}

Question :
{question}
"""

In [110]:
reponse = llm.invoke(prompt)

In [111]:
print(reponse.content)

L'information concernant le recrutement d'alternants par la Caisse des dépôts n'a pas été trouvée dans les documents fournis.
